# Ours-YOLO — Train trên Kaggle (VisDrone2019)

Notebook clone code trực tiếp từ GitHub thay vì copy thủ công — luôn chạy đúng phiên bản
mới nhất trên `main`. Sửa code thì push lên GitHub rồi rerun/Save Version, không sửa trực
tiếp trong Kaggle. Dataset tự tải qua `gdown` — không cần tạo Kaggle Dataset/Add Input thủ công.

**Mỗi lần chạy = đúng 1 dòng trong `results/SOTA-SLIM.xlsx`.** Chọn dòng nào bằng biến
`CONFIG_FILE` ở cell CẤU HÌNH; tên run tự đặt theo tên file config (vd `exp02`), nên
không thể lẫn kết quả giữa các thí nghiệm.

## Chuẩn bị (làm 1 lần)
1. **Settings → Accelerator → GPU P100 hoặc T4x2** — cả 2 đều dùng được, xem `DEVICE` ở cell
   CẤU HÌNH (`"0"` = 1 GPU, `"0,1"` = cả 2 GPU T4 — code đã hỗ trợ multi-GPU/DDP).
2. **Settings → Internet → ON** (bắt buộc để `git clone`, `pip install`, và tải dataset qua `gdown` hoạt động).
3. Nếu repo GitHub là **private**: sửa `GIT_URL` thành `https://<token>@github.com/...` (Personal
   Access Token) hoặc dùng Kaggle Secrets. Repo public thì bỏ qua bước này.

## ⚠️ Đồng nhất recipe
`IMGSZ` và `EPOCHS` **phải khớp** với các run đã có, nếu không cả bảng mất giá trị so sánh.
Đọc lại recipe của một checkpoint cũ bằng:
```python
import torch; print(torch.load("best.pt", map_location="cpu", weights_only=False)["train_args"])
```

## Thời gian
~6-8 phút/epoch @960 trên P100 (1 GPU); @640 nhanh hơn đáng kể. T4x2 (`DEVICE="0,1"`)
nhanh hơn nhưng cần đủ VRAM cho cả 2 GPU cùng lúc — nếu OOM, giảm `BATCH` hoặc dùng 1 GPU.
Muốn train dài hơn: sau mỗi phiên **Save Version** để giữ `SLIM/runs/`, phiên sau đặt
`RESUME_FROM` trỏ tới `last.pt` (attach output của version trước làm Input).


In [ ]:
# ===================== CẤU HÌNH =====================
GIT_URL  = "https://github.com/trong5nhan6/SLIM.git"
REPO_DIR = "/kaggle/working/SLIM"

# ---- CONFIG_FILE: mỗi file = đúng 1 dòng trong results/SOTA-SLIM.xlsx ----
# Notebook chạy `train.py --config <file>`; model + loss lấy hết TỪ FILE ĐÓ.
# Số params/GFLOPs@640 là bản CHƯA fuse; latency xem block EFFICIENCY METRICS sau khi train trên T4.
CONFIG_FILE = "configs/exp02.yaml"
# --- Bảng 2 — đổi block (2x2: upsampler x block) ---
# CONFIG_FILE = "configs/exp01.yaml"   # 1  DySample + C3k2Ghost (SLIM)        0.560M / 6.31G  [đã chạy]
# CONFIG_FILE = "configs/exp02.yaml"   # 2  DySample + C3k2Rep                 0.945M / 9.78G
# CONFIG_FILE = "configs/exp03.yaml"   # 3  bilinear + C3k2Ghost               0.553M / 6.33G
# CONFIG_FILE = "configs/exp04.yaml"   # 4  bilinear + C3k2Rep = Ours-N        0.938M / 9.79G  [đã chạy]
# --- Bảng 3 — scale ---
# CONFIG_FILE = "configs/exp06.yaml"   # 6  Ours-S                             2.373M / 23.65G
# CONFIG_FILE = "configs/exp07.yaml"   # 7  Ours-M            (tùy chọn)       4.208M / 41.59G
# CONFIG_FILE = "configs/exp08.yaml"   # 8  YOLOv11n from-scratch              2.592M / 6.45G  [đã chạy]
# CONFIG_FILE = "configs/exp09.yaml"   # 9  YOLOv11s from-scratch              9.432M / 21.57G [đã chạy]
# CONFIG_FILE = "configs/exp10.yaml"   # 10 YOLOv11m from-scratch (tùy chọn)   20.06M / 68.23G
# --- Bảng 4 — gỡ từng thành phần khỏi Ours-N ---
# CONFIG_FILE = "configs/exp12.yaml"   # 12 - LSCDetect (stock Detect)         1.362M / 16.30G
# CONFIG_FILE = "configs/exp13.yaml"   # 13 - nhánh P2 (head P3+P4)            0.812M / 5.86G
# CONFIG_FILE = "configs/exp14.yaml"   # 14 - NWD (CIoU thuần, chỉ đổi loss)   0.938M / 9.79G
# CONFIG_FILE = "configs/exp15.yaml"   # 15 - attention (EMA+C2PSA+ECA)        0.872M / 9.33G
# CONFIG_FILE = "configs/exp16.yaml"   # 16 - SPDConv (Conv stride-2)          1.108M / 10.83G
# CONFIG_FILE = "configs/exp17.yaml"   # 17 head GroupNorm -> BatchNorm        0.938M / 9.83G
# CONFIG_FILE = "configs/exp18.yaml"   # 18 - P2 + P5 = pyramid P3/P4/P5       3.001M / 7.77G

DEVICE       = "0"         # "0" = 1 GPU; "0,1" = ca 2 GPU T4 tren Kaggle (DDP)
IMGSZ        = 640         # PHAI khop voi cac run truoc, neu khong bang so sanh mat gia tri
EPOCHS       = 100         # PHAI khop voi cac run truoc (xem o tren cach doc tu train_args)
BATCH        = 32          # Voi DEVICE="0,1" day la TONG batch chia cho 2 GPU (ultralytics tu chia)
RUN_NAME     = None        # None = tu dat theo ten CONFIG_FILE (vd "exp02")
TEST_EVERY   = 20          # 0 = tat; log mAP tap test moi N epoch (chi giam sat, khong chon best.pt)
RESUME_FROM  = ""          # vd "/kaggle/input/<output-version-truoc>/SLIM/runs/detect/exp02/weights/last.pt"


In [ ]:
import os

%pip install -q "ultralytics>=8.3,<8.4"

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone --depth 1 {GIT_URL} {REPO_DIR}
%cd {REPO_DIR}

!pip install -q -r requirements.txt

import ultralytics, torch
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__,
      "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")


In [ ]:
# sanity check nhanh trước khi tốn GPU: custom module + patch parse_model có build đúng không
!pytest tests/ -q


In [ ]:
# ============ Tải VisDrone từ Google Drive + tạo data yaml ============
!pip install -q gdown
!mkdir -p datasets
!gdown 1_DTApN9Vwhex4v2m_39aTmNVF4wUA5E- -O datasets/VisDrone.zip
!unzip -q -o datasets/VisDrone.zip -d datasets

import glob
from pathlib import Path

patterns = ["datasets/VisDrone/images/train",
            "datasets/*/images/train",
            "datasets/images/train"]
hits = [p for pat in patterns for p in glob.glob(pat)]
assert hits, "Khong tim thay images/train duoi datasets/ sau khi giai nen — kiem tra lai file zip/Drive ID"
DATA_ROOT = str(Path(sorted(set(hits))[0]).parents[1])
print("Dataset root:", DATA_ROOT)

names = ["pedestrian", "people", "bicycle", "car", "van",
         "truck", "tricycle", "awning-tricycle", "bus", "motor"]
# Tên riêng, không phải data/visdrone.yaml của repo (file đó trỏ path Windows local).
DATA_YAML = "visdrone_kaggle.yaml"
yaml_text = (
    f"path: {DATA_ROOT}\n"
    "train: images/train\n"
    "val: images/val\n"
    "test: images/test\n"
    "names:\n" + "".join(f"  {i}: {n}\n" for i, n in enumerate(names))
)
Path(DATA_YAML).write_text(yaml_text)
print(yaml_text)


In [ ]:
# ===================== TRAIN =====================
# train.py merge: configs/base.yaml <- CONFIG_FILE <- cac flag duoi day.
# Model + loss lay het tu CONFIG_FILE, nen KHONG truyen --idea/--model/--nwd-*.
from pathlib import Path

assert CONFIG_FILE, "Dat CONFIG_FILE o cell CAU HINH (vd 'configs/exp02.yaml')"
ACTUAL_RUN_NAME = RUN_NAME or Path(CONFIG_FILE).stem  # vd "exp02"

cmd = f"python train.py --config {CONFIG_FILE} --data {DATA_YAML} --imgsz {IMGSZ} "
cmd += f"--epochs {EPOCHS} --batch {BATCH} --device {DEVICE} --name {ACTUAL_RUN_NAME} "
cmd += f"--test-every {TEST_EVERY}"
if RESUME_FROM:
    cmd += f" --model {RESUME_FROM}"

print(cmd)
!{cmd}


In [ ]:
# ===================== VALIDATE + XUAT KET QUA =====================
import shutil
from pathlib import Path

# runs/detect/{ACTUAL_RUN_NAME} co the bi ultralytics doi ten (vd tenrun2) neu thu muc
# da ton tai tu 1 lan chay truoc trong cung session (exist_ok=False mac dinh) —
# lay thu muc khop ACTUAL_RUN_NAME* moi cap nhat gan nhat thay vi hardcode dung ten.
run_dirs = sorted(Path("runs/detect").glob(f"{ACTUAL_RUN_NAME}*"), key=lambda p: p.stat().st_mtime)
assert run_dirs, f"Khong tim thay runs/detect/{ACTUAL_RUN_NAME}* — train da chay xong chua?"
run_dir = run_dirs[-1]
if run_dir.name != ACTUAL_RUN_NAME:
    print(f"[canh bao] dung {run_dir.name} thay vi {ACTUAL_RUN_NAME} (ultralytics da doi ten do trung thu muc)")

# val split: doi chieu voi mAP train log (best.pt duoc chon theo split nay).
# test split: benchmark that su (test-dev), + efficiency metrics (params/GFLOPs/latency/FPS)
# in ra tu val.py — day moi la con so nen bao cao/so sanh giua cac idea.
for split in ("val", "test"):
    val_cmd = (
        f"python val.py --weights {run_dir}/weights/best.pt "
        f"--data {DATA_YAML} --imgsz {IMGSZ} --split {split}"
    )
    print(f"\n===== split={split} =====")
    print(val_cmd)
    !{val_cmd}

best = run_dir / "weights" / "best.pt"
shutil.copy(best, f"/kaggle/working/best_{ACTUAL_RUN_NAME}.pt")
print(f"\nDa copy best.pt -> /kaggle/working/best_{ACTUAL_RUN_NAME}.pt (tai ve o tab Output)")

# logs/<name>.log + test_metrics.csv nam trong repo clone (/kaggle/working/SLIM/...) —
# chi ton tai trong Output neu ban Save Version ca thu muc do. Copy rieng ra /kaggle/working
# de chac chan giu duoc ngay ca khi ban chi tai best_*.pt tu tab Output.
log_src = Path("logs") / f"{ACTUAL_RUN_NAME}.log"
csv_src = run_dir / "test_metrics.csv"
if log_src.exists():
    shutil.copy(log_src, f"/kaggle/working/{log_src.name}")
    print(f"Da copy {log_src} -> /kaggle/working/{log_src.name}")
else:
    print(f"[canh bao] khong thay {log_src} — train da chay xong chua?")
if csv_src.exists():
    shutil.copy(csv_src, f"/kaggle/working/test_metrics_{ACTUAL_RUN_NAME}.csv")
    print(f"Da copy {csv_src} -> /kaggle/working/test_metrics_{ACTUAL_RUN_NAME}.csv")


## Sau khi train

- **Save Version** (Save & Run All hoặc Quick Save) để giữ `SLIM/runs/` + `best_*.pt` trong Output.
- Train tiếp phiên sau: attach Output của version trước làm Input, đặt
  `RESUME_FROM = "/kaggle/input/<slug>/SLIM/runs/detect/<ten-run-cu>/weights/last.pt"`
  (`train.py --model` nhận thẳng `.pt` để finetune tiếp — không phải resume optimizer state
  đúng nghĩa, hợp lý vì mỗi phiên Kaggle là một lượt optimizer/cos_lr schedule mới).
- **Tên run** tự đặt theo tên `CONFIG_FILE` (`RUN_NAME = None` mặc định) — chạy
  `configs/exp02.yaml` thì ra `runs/detect/exp02/`, không thể lẫn giữa các thí nghiệm.
- Cell VALIDATE tự chạy cả `--split val` (đối chiếu với training log, `best.pt` được chọn theo
  split này) lẫn `--split test` (benchmark thật). `val.py` in ra đủ 10 metric của bảng, kết
  bằng một dòng **PAPER TABLE ROW** dán thẳng vào `results/SOTA-SLIM.xlsx`:
  `P(%), R(%), mAP50, mAP50-95, Params(M), GFLOPs, Size(MB), FPS, Epochs, imgsz`
- Cột **Model size** trong bảng dùng dòng `Checkpoint file size (.pt)` (dung lượng file thật,
  fp16) — KHÔNG dùng dòng `Model size (fp32, theoretical)`, số đó gấp đôi thực tế.
- `logs/<run>.log` + `test_metrics.csv` được copy ra `/kaggle/working/` để không mất khi chỉ
  tải riêng `best_*.pt` từ tab Output.
- Code luôn lấy từ `main` trên GitHub lúc `git clone`/`git pull` ở cell setup — sửa code
  (kể cả `cfg/*.yaml`, `configs/*.yaml`) thì push lên GitHub rồi rerun cell đó, không sửa
  trực tiếp trong Kaggle.
